# ASG Airlines Pipeline

Ingestion -> cleaning -> PII masking -> MySQL load,

In [1]:
import sys
sys.path.append("../src")

import logging
from pathlib import Path

import pandas as pd
import ingestion
import flights_cleaning
import passengers_cleaning
import bookings_cleaning
import payments_cleaning
import pii_masking

# Logging alongside the existing print() calls, not replacing them -- the
# prints are readable narrative for this notebook, logging separately keeps
# an audit trail (stage start/end, row counts, failures) in logs/pipeline.log
Path("../logs").mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("../logs/pipeline.log"),
        logging.StreamHandler(),
    ],
)
logging.getLogger("mysql.connector").setLevel(logging.WARNING)  # quiet its internal auth-plugin chatter
logger = logging.getLogger("airlines_pipeline")
logger.info("Pipeline run started")

2026-09-11 10:42:19,768 INFO airlines_pipeline: Pipeline run started


## 1. Ingestion

Loads all four raw sheets from `UseCase - Airlines.xlsx`. `passengers` is read with
`dtype={"aadhaar_id": str}` -- pandas would otherwise infer that all-digit column as `int64` and
silently drop leading zeros.

In [2]:
raw = ingestion.load_all()
for name, df in raw.items():
    print(name, df.shape)

2026-09-11 10:42:20,561 INFO ingestion: Loaded sheet 'flights': 1020 rows, 7 columns


2026-09-11 10:42:20,718 INFO ingestion: Loaded sheet 'passengers': 1039 rows, 9 columns


2026-09-11 10:42:20,930 INFO ingestion: Loaded sheet 'bookings': 1000 rows, 9 columns


2026-09-11 10:42:21,067 INFO ingestion: Loaded sheet 'payments': 1000 rows, 4 columns


flights (1020, 7)
passengers (1039, 9)
bookings (1000, 9)
payments (1000, 4)


### Ingestion-time data quality checks

Null counts and duplicate-key counts for each sheet.

In [3]:
ingestion.profile(raw["flights"], key_col="flight_id")

flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64
exact duplicate rows: 15
duplicate flight_id: 32


In [4]:
ingestion.profile(raw["passengers"], key_col="passenger_id")

passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64
exact duplicate rows: 0
duplicate passenger_id: 75


In [5]:
ingestion.profile(raw["bookings"], key_col="booking_id")

booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
exact duplicate rows: 0
duplicate booking_id: 0


In [6]:
ingestion.profile(raw["payments"], key_col="payment_id")

payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
exact duplicate rows: 0
duplicate payment_id: 0


## 2. Cleaning


### 2.1 flights

In [7]:
logger.info("Cleaning stage started: flights")
try:
    flights_clean = flights_cleaning.clean_flights(raw["flights"])
    logger.info(
        "Cleaning stage succeeded: flights (%d -> %d rows)",
        len(raw["flights"]), len(flights_clean)
    )
except Exception:
    logger.exception("Cleaning stage failed: flights")
    raise

2026-09-11 10:42:21,135 INFO airlines_pipeline: Cleaning stage started: flights


2026-09-11 10:42:21,173 INFO airlines_pipeline: Cleaning stage succeeded: flights (1020 -> 1005 rows)


flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64
15
0
flight_id          0
airline           39
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64
69
0
0
airline
IndiGo       273
Air India    255
SpiceJet     247
Vistara      230
Name: count, dtype: int64
flight_id
6F    273
AI    255
SJ    247
UK    230
Name: count, dtype: int64
flight_id
5    1005
Name: count, dtype: int64
Colliding flight_id values: ['6F250']
           departure_time            arrival_time  duration
0 2026-04-20 23:38:41.701 2026-04-21 02:32:41.701  02:54:00
1 2026-04-20 23:35:41.703 2026-04-21 01:23:41.703  01:48:00
2 2026-04-20 23:26:41.702 2026-04-21 01:11:41.702  01:45:00
3 2026-04-20 23:07:41.704 2026-04-21 01:43:41.704  02:36:00
4 2026-04-20 23:05:41.703 2026-04-21 04:04:41.703  04:59:00
5 2026-04-20 23:05:41.703 2026-04-21 01:24:41.

### 2.2 passengers

In [8]:
logger.info("Cleaning stage started: passengers")
try:
    passengers_clean = passengers_cleaning.clean_passengers(raw["passengers"])
    logger.info(
        "Cleaning stage succeeded: passengers (%d -> %d rows)",
        len(raw["passengers"]), len(passengers_clean)
    )
except Exception:
    logger.exception("Cleaning stage failed: passengers")
    raise

2026-09-11 10:42:21,186 INFO airlines_pipeline: Cleaning stage started: passengers


passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64
0
75
0
0
(1000, 9)
passenger_id     0
first_name       0
last_name        0
age              0
gender           0
email            0
phone            0
aadhaar_id       0
date_of_birth    0
dtype: int64
0
count    1000.000000
mean       43.314000
std        25.834102
min         1.000000
25%        21.000000
50%        44.000000
75%        64.000000
max        89.000000
Name: age, dtype: float64
Invalid age values: 0
Age mismatches: 0
Empty DataFrame
Columns: [passenger_id, age, date_of_birth, age_by_year]
Index: []
gender
M    527
F    473
Name: count, dtype: int64
Invalid email values: 0
Invalid phone values: 0
Invalid Aadhaar values: 0
Invalid passenger_id values: 0
  first_name   last_name
0     Vivaan  Chatterjee
1    Krishna       Reddy
2       Myra       Naidu
3       Myra      Mishra
4   

2026-09-11 10:42:21,244 INFO airlines_pipeline: Cleaning stage succeeded: passengers (1039 -> 1000 rows)


### 2.3 bookings

In [9]:
logger.info("Cleaning stage started: bookings")
try:
    bookings_clean = bookings_cleaning.clean_bookings(raw["bookings"])
    logger.info(
        "Cleaning stage succeeded: bookings (%d -> %d rows)",
        len(raw["bookings"]), len(bookings_clean)
    )
except Exception:
    logger.exception("Cleaning stage failed: bookings")
    raise

2026-09-11 10:42:21,297 INFO airlines_pipeline: Cleaning stage started: bookings


booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
0
0
0
booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
0    B1000
1    B1001
2    B1002
3    B1003
4    B1004
Name: booking_id, dtype: str
Invalid booking_id values: 0
Invalid passenger_id values: 0
Invalid flight_id values: 0
datetime64[us]
0   2025-06-14 11:37:36.951
1   2025-11-02 11:37:36.951
2   2025-08-25 11:37:36.951
3   2025-12-30 11:37:36.951
4   2025-10-02 11:37:36.951
5   2025-08-31 11:37:36.951
6   2025-06-29 11:37:36.951
7   2026-03-12 11:37:36.951
8   2025-09-07 11

2026-09-11 10:42:21,354 INFO airlines_pipeline: Cleaning stage succeeded: bookings (1000 -> 1000 rows)


0
(1000, 9)


### 2.4 payments

In [10]:
logger.info("Cleaning stage started: payments")
try:
    payments_clean = payments_cleaning.clean_payments(raw["payments"])
    logger.info(
        "Cleaning stage succeeded: payments (%d -> %d rows)",
        len(raw["payments"]), len(payments_clean)
    )
except Exception:
    logger.exception("Cleaning stage failed: payments")
    raise

2026-09-11 10:42:21,372 INFO airlines_pipeline: Cleaning stage started: payments


2026-09-11 10:42:21,409 INFO airlines_pipeline: Cleaning stage succeeded: payments (1000 -> 1000 rows)


payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
0
0
0
payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
0    PAY1000
1    PAY1001
2    PAY1002
3    PAY1003
4    PAY1004
Name: payment_id, dtype: str
Invalid payment_id values: 0
Invalid booking_id values: 0
0      9883.49
1      8457.96
2      6495.37
3      5079.38
4     12518.31
5      7319.71
6      8020.26
7       8291.6
8      7277.14
9      4585.24
10     2684.77
11     2379.52
12    11604.48
13     4129.39
14     1779.42
15    12165.24
16     3156.86
17     9752.05
18         NaN
19     INVALID
Name: amount, dtype: object
object
float64
78
count      922.000000
mean      8009.916464
std       4022.055392
min       1002.590000
25%       4623.495000
50%       8027.125000
75%      11288.155000
max      14992.950000
Name: amount, dtype: float64
Invalid payment amounts: 78
    payment_id booking_id  amount
18     PAY1018      B1798     NaN
1

### Saving cleaned output


In [11]:
flights_clean.to_excel("../data/cleaned/cleaned_data_flights.xlsx", index=False)
passengers_clean.to_excel("../data/cleaned/cleaned_data_passengers.xlsx", index=False)
bookings_clean.to_excel("../data/cleaned/cleaned_data_bookings.xlsx", index=False)
payments_clean.to_excel("../data/cleaned/cleaned_data_payments.xlsx", index=False)
payments_clean.to_csv("../data/cleaned/payments.csv", index=False)

print("Saved 4 cleaned .xlsx files + payments.csv")

Saved 4 cleaned .xlsx files + payments.csv


## 3. PII masking

Only `passengers` and `bookings` hold PII
Masking technique is chosen per column by its actual re-identification risk: `first_name` is
kept (a weak identifier alone), `last_name` is reduced to an initial (the full name together is
what's identifying), `email`/`phone`/`aadhaar_id`/`passport_number` are partially masked, and
`emergency_contact_name` is fully redacted since it has no analytical value at all.

In [12]:
logger.info("PII masking stage started")
try:
    passengers_analytics = pii_masking.mask_passengers(passengers_clean)
    bookings_analytics = pii_masking.mask_bookings(bookings_clean)

    passengers_analytics.to_excel("../data/cleaned/passengers_analytics.xlsx", index=False)
    bookings_analytics.to_excel("../data/cleaned/bookings_analytics.xlsx", index=False)
    logger.info("PII masking stage succeeded")
except Exception:
    logger.exception("PII masking stage failed")
    raise

print(passengers_analytics[["first_name", "last_name", "email", "phone", "aadhaar_id"]].head(3))
print(bookings_analytics[["passport_number", "emergency_contact_name", "emergency_contact_phone"]].head(3))

2026-09-11 10:42:23,844 INFO airlines_pipeline: PII masking stage started


2026-09-11 10:42:25,083 INFO airlines_pipeline: PII masking stage succeeded


  first_name last_name              email       phone    aadhaar_id
0     Vivaan        C.    v****@gmail.com  ******3790  XXXXXXXX6001
1    Krishna        R.  k****@hotmail.com  ******2297  XXXXXXXX2654
2       Myra        N.  m****@outlook.com  ******5092  XXXXXXXX8161
  passport_number emergency_contact_name emergency_contact_phone
0        XXXX5887                 MASKED              ******5128
1        XXXX2012                 MASKED              ******8662
2        XXXX7659                 MASKED              ******8220


## 4. Loading into MySQL

`payments` loads via `mysql.connector`, It correctly inserts `NULL` for the 78 rows with an
invalid/missing amount, not `0`.

**Note:** `passengers`/`bookings` were reloaded into MySQL from the masked files via a
staging-table `LOAD DATA LOCAL INFILE` + `UPDATE ... JOIN`, done interactively in MySQL
Workbench. That step has no saved script, so it isn't repeated here -- this cell only covers
the payments load.

In [13]:
import os
import mysql.connector

logger.info("MySQL load stage started: payments")

connection = None
cursor = None
try:
    connection = mysql.connector.connect(
        option_files=os.path.expanduser("~/.my.cnf"),
        database="airlines_dw"
    )
    cursor = connection.cursor()

    cursor.execute("DELETE FROM payments")

    insert_query = """
    INSERT INTO payments (
        payment_id,
        booking_id,
        amount,
        payment_method
    )
    VALUES (%s, %s, %s, %s)
    """

    for _, row in payments_clean.iterrows():
        amount = (
            None
            if pd.isna(row['amount'])
            else float(row['amount'])
        )
        cursor.execute(
            insert_query,
            (row['payment_id'], row['booking_id'], amount, row['payment_method'])
        )

    connection.commit()

    cursor.execute("SELECT COUNT(*) FROM payments")
    total = cursor.fetchone()[0]
    print("Payments loaded into MySQL:", total)

    cursor.execute("SELECT COUNT(*) FROM payments WHERE amount IS NULL")
    null_count = cursor.fetchone()[0]
    print("Payments with missing amount:", null_count)

    logger.info("MySQL load stage succeeded: %d rows loaded, %d null amounts", total, null_count)

except Exception:
    logger.exception("MySQL load stage failed -- rolling back")
    if connection is not None:
        connection.rollback()
    raise

finally:
    if cursor is not None:
        cursor.close()
    if connection is not None:
        connection.close()

2026-09-11 10:42:25,260 INFO airlines_pipeline: MySQL load stage started: payments


2026-09-11 10:42:26,487 INFO airlines_pipeline: MySQL load stage succeeded: 1000 rows loaded, 78 null amounts


Payments loaded into MySQL: 1000
Payments with missing amount: 78


## Pipeline complete

Raw workbook -> cleaned (`data/cleaned/`) -> masked (`*_analytics.xlsx`) -> loaded into
`airlines_dw.payments`. KPI queries live in `sql/airlines_analysis.sql`; the Power BI dashboard
in `powerbi/` connects live to `airlines_dw`.